# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 clinical oncology dataset using the `mlcroissant` library. The dataset is structured with a Croissant schema and is retrieved live via its URL.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, their `@id`s, and get a summary of the dataset schema.

In [ ]:
# List all record set @ids and names
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs['name']}")

# For each record set, list its field @ids and names
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # ensure list
    for field in fields:
        print(f"  Field @id: {field['@id']} | name: {field['name']} | dataType: {field.get('dataType', 'unknown')}")

### Example: Preview the records in the main record set

Replace `<record_set_id>` with the `@id` of the record set you want to preview.

In [ ]:
# Preview sample records for a record set by @id
main_record_set_id = None
for rs in record_sets:
    if 'colorectal' in rs['name'].lower() or True:
        main_record_set_id = rs['@id']
        break
# main_record_set_id example: 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json:PatientDataTable'
print(f"\nPreview records from record set @id: {main_record_set_id}")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i > 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using their `@id`s.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
for rs in record_sets:
    record_set_id = rs['@id']
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {df.shape}")


### Examine columns in the main data table

Columns correspond to field `@id`s.

In [ ]:
# Display main record set columns and head
df_main = dataframes[main_record_set_id]
print("Columns (field @ids):", df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.

**Note:** All fields are referenced by their `@id`.

In [ ]:
# Example: Select numeric fields from schema to analyze
# We'll search the schema for candidate numeric field @ids
numeric_field_id = None
group_field_id = None

# Scan available fields for possible numeric fields
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        # Typically, field is a list of field dicts
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            dt = f.get('dataType', '').lower()
            if dt in ('integer', 'float', 'number', 'schema:number', 'schema:integer', 'schema:float'):
                numeric_field_id = f['@id']
                break
        # As an example, set group_field_id to anatomical_location or similar variable
        for f in fields:
            if 'anatomical' in f['name'].lower():
                group_field_id = f['@id']
                break
if not numeric_field_id:
    numeric_field_id = df_main.select_dtypes(include='number').columns[0]
if not group_field_id:
    group_field_id = df_main.columns[1]  # just pick the second column as an example

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

threshold = df_main[numeric_field_id].mean()  # use mean as threshold example
# Filter records with values above threshold
filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the group_field_id, compute mean
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id}: (mean of {numeric_field_id})")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_main[numeric_field_id], kde=True, bins=20)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot of numeric field by group
if group_field_id in df_main.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have explored the FAIR^2 dataset using the `mlcroissant` library, extracted clinical record tables, identified numerical and categorical fields by their `@id`, and performed basic filtering, normalization, grouping, and visualization—all referencing elements by their Croissant `@id` fields. This workflow supports reproducible, schema-consistent exploration of real-world biomedical data.